加载指定目录1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json以_privacy_policy_links结尾的csv，读取里面apk_name和privacy_policy_url，
访问privacy_policy_url，下载privacy_policy的正文内容为markdown，保存到1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_google_play\{apkname}，也用csv文件保存下载结果

In [ ]:
# {f_position}\{s_position}
f_position = "AA2_second_batch"
s_position = "AA6_sisth_100_batch" # AA6_sisth_100_batch, AA7_seventh_100_batch # AA3_third_100_batch, AA4_forth_100_batch, AA5_fifth_100_batch

In [141]:
from pathlib import Path
import re
import time
import random
from html import unescape

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# =========================
# paths
# =========================
# IN_DIR = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json")

# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_waybackmachine_md.csv
IN_DIR_PATH = Path(rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_waybackmachine_md.csv")

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\apkitself
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\apkitself
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\apkitself
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\apkitself
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\apkitself
OUT_DIR = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\apkitself")
OUT_CSV = OUT_DIR / "pp_apkitself_download_results.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [142]:
# =========================
# requests session
# =========================
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)


# =========================
# helpers
# =========================
def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name)
    name = re.sub(r"\s+", "_", name)
    return name[:200] if name else "unknown_apk"


def html_to_text(html: str) -> str:
    html = re.sub(r"(?is)<script.*?>.*?</script>", " ", html)
    html = re.sub(r"(?is)<style.*?>.*?</style>", " ", html)
    html = re.sub(r"(?i)</p>|<br\s*/?>|</div>|</li>|</tr>|</h[1-6]>", "\n", html)
    html = re.sub(r"(?i)<li[^>]*>", "- ", html)
    text = re.sub(r"(?s)<[^>]+>", " ", html)
    text = unescape(text)
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    return text.strip()


def extract_main_html(html: str) -> str:
    """
    尽量抽正文区域；抽不到就返回原始 html
    """
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")

        # 去掉明显噪音
        for tag in soup(["script", "style", "noscript", "svg", "iframe", "footer", "nav"]):
            tag.decompose()

        # 优先正文容器
        candidates = []

        for selector in [
            "main",
            "article",
            '[role="main"]',
            ".content",
            ".main-content",
            ".post-content",
            ".entry-content",
            ".page-content",
            ".policy",
            ".privacy-policy",
        ]:
            for node in soup.select(selector):
                txt = node.get_text(" ", strip=True)
                if len(txt) > 500:
                    candidates.append((len(txt), str(node)))

        if candidates:
            candidates.sort(reverse=True)
            return candidates[0][1]

        # 回退：找文本最多的 div/section
        blocks = []
        for node in soup.find_all(["div", "section"]):
            txt = node.get_text(" ", strip=True)
            if len(txt) > 1000:
                blocks.append((len(txt), str(node)))

        if blocks:
            blocks.sort(reverse=True)
            return blocks[0][1]

        body = soup.body
        if body:
            return str(body)

        return html

    except Exception:
        return html


def html_to_markdown(html: str) -> str:
    """
    优先 markdownify；失败则退化为纯文本
    """
    main_html = extract_main_html(html)

    try:
        from markdownify import markdownify as md
        md_text = md(main_html, heading_style="ATX")
        md_text = unescape(md_text)
        md_text = re.sub(r"\n{3,}", "\n\n", md_text).strip()
        return md_text
    except Exception:
        return html_to_text(main_html)


def fetch_and_save_markdown(apk_name: str, version: str, url: str):
    out_path = OUT_DIR / f"{apk_name}_{version}.md"

    try:
        resp = session.get(url, timeout=(10, 90), allow_redirects=True)
        status = resp.status_code

        if status != 200:
            return {
                "success": False,
                "http_status": status,
                "final_url": resp.url if hasattr(resp, "url") else "",
                "saved_path": "",
                "content_length": 0,
                "error": f"http_{status}",
            }

        resp.encoding = resp.apparent_encoding or resp.encoding
        # md_text = html_to_markdown(resp.text)

        # if not md_text or len(md_text.strip()) < 50:
        #     return {
        #         "success": False,
        #         "http_status": status,
        #         "final_url": resp.url,
        #         "saved_path": "",
        #         "content_length": 0,
        #         "error": "content_too_short",
        #     }

        # with open(out_path, "w", encoding="utf-8") as f:
        #     f.write(md_text)

        return {
            "success": True,
            "http_status": status,
            "final_url": resp.url,
            "saved_path": str(out_path),
            # "content_length": len(md_text),
            "error": "",
        }

    except Exception as e:
        return {
            "success": False,
            "http_status": "",
            "final_url": "",
            "saved_path": "",
            "content_length": 0,
            "error": f"{type(e).__name__}: {e}",
        }

In [143]:
# =========================
# load csv
# =========================
df_in = pd.read_csv(IN_DIR_PATH)
df_in.head(), df_in.shape

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [144]:
df_apkitself_in = df_in[df_in["source"]=="apkitself"]
df_apkitself_in.head(), df_apkitself_in.shape

(                                        apk_name  version  \
 8                      com.polyverse.bricks.game      122   
 9                      com.polyverse.bricks.game      123   
 20  com.productivity.musicdj.mixer2.sound.effect      123   
 21  com.productivity.musicdj.mixer2.sound.effect      124   
 26                          com.punjabimatrimony      335   
 
                                             json_file     source  \
 8             com.polyverse.bricks.game-122_urls.json  apkitself   
 9             com.polyverse.bricks.game-123_urls.json  apkitself   
 20  com.productivity.musicdj.mixer2.sound.effect-1...  apkitself   
 21  com.productivity.musicdj.mixer2.sound.effect-1...  apkitself   
 26                 com.punjabimatrimony-335_urls.json  apkitself   
 
                                           privacy_url  \
 8          https://www.riveroll.top/privacy-policy-2/   
 9          https://www.riveroll.top/privacy-policy-2/   
 20  https://pansyspinger.000webhost

In [145]:
required_cols = {"apk_name", "privacy_url"}
missing = required_cols - set(df_apkitself_in.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df_apkitself_in["apk_name"] = df_apkitself_in["apk_name"].astype(str).str.strip()
df_apkitself_in["privacy_url"] = df_apkitself_in["privacy_url"].astype(str).str.strip()

# 清理空值
df_apkitself_in = df_apkitself_in[
    (df_apkitself_in["apk_name"] != "") &
    (df_apkitself_in["privacy_url"] != "") &
    (df_apkitself_in["privacy_url"].str.lower() != "nan")
].copy()

# 同一个 apk_name 保留第一条非空链接
# df_apkitself_in = df_apkitself_in.drop_duplicates(subset=["apk_name"], keep="first").reset_index(drop=True)
df_apkitself_in.shape

(17, 8)

In [146]:
# =========================
# run
# =========================
from marshal import version


rows = []
total = len(df_apkitself_in)

for i, row in df_apkitself_in.iterrows():
    apk_name = row["apk_name"]
    version = row["version"]
    privacy_policy_url = row["privacy_url"]

    result = fetch_and_save_markdown(apk_name, version, privacy_policy_url)

    rows.append({
        "apk_name": apk_name,
        "version": version,
        "privacy_policy_url": privacy_policy_url,
        "apkitself_url": result["final_url"],
        "http_status": result["http_status"],
        "success": result["success"],
        "saved_path": result["saved_path"],
        # "content_length": result["content_length"],
        "error": result["error"],
    })

    if (i + 1) % 20 == 0 or (i + 1) == total:
        print(f"Progress: {i + 1}/{total}")

    time.sleep(random.uniform(0.8, 1.6))


# =========================
# save result csv
# =========================
df_out = pd.DataFrame(rows)
df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("Done.")
print("Output CSV:", OUT_CSV)
print("Downloaded:", int(df_out["success"].sum()), "/", len(df_out))

Done.
Output CSV: 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA2_second_batch\AA4_forth_100_batch\apkitself\pp_apkitself_download_results.csv
Downloaded: 12 / 17


In [173]:
# OUT_CSV

更新表

然后在apk_versions_summary_waybackmachine里source为apkitself的行的apkname，在下载结果里更新到表里，下载结果为1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\apkitself\pp_apkitself_download_results.csv

下载成功的可以将apk_versions_summary.csv的对应的apkname和version和source为apkitself哪一行的source改为apkitself，把apkitself_url加入

In [174]:
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch\apk_versions_summary_waybackmachine_md.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_waybackmachine_md.csv"
maintainess_apk_summary_df = pd.read_csv(apk_summary_df_path)
maintainess_apk_summary_df.head(), maintainess_apk_summary_df.shape

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [175]:
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\apkitself\pp_apkitself_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\apkitself\pp_apkitself_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\apkitself\pp_apkitself_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\apkitself\pp_apkitself_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\apkitself\pp_apkitself_download_results.csv
csv_path = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\apkitself\pp_apkitself_download_results.csv")
md_results_df = pd.read_csv(csv_path, dtype={"apk_name": str, "version": str})
md_results_df.head(), md_results_df.shape

(                                       apk_name version  \
 0                     com.polyverse.bricks.game     122   
 1                     com.polyverse.bricks.game     123   
 2  com.productivity.musicdj.mixer2.sound.effect     123   
 3  com.productivity.musicdj.mixer2.sound.effect     124   
 4                          com.punjabimatrimony     335   
 
                                   privacy_policy_url  \
 0         https://www.riveroll.top/privacy-policy-2/   
 1         https://www.riveroll.top/privacy-policy-2/   
 2  https://pansyspinger.000webhostapp.com/privacy...   
 3  https://pansyspinger.000webhostapp.com/privacy...   
 4  https://apps.bharatmatrimony.com/appassuredcon...   
 
                                        apkitself_url  http_status  success  \
 0         https://www.riveroll.top/privacy-policy-2/        200.0     True   
 1         https://www.riveroll.top/privacy-policy-2/        200.0     True   
 2                                                NaN    

In [176]:
# 0. 先复制，避免污染原表
left = maintainess_apk_summary_df.copy()
right = md_results_df.copy()
left.head(), right.head()

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [177]:
# 1. 统一键字段类型
for df in [left, right]:
    df["apk_name"] = df["apk_name"].astype(str).str.strip()
    df["version"] = df["version"].astype(str).str.strip()

In [178]:
# 2. 只保留 apkitself_url不为空 的行
src = right.loc[
    right["apkitself_url"].notna() &
    (right["apkitself_url"].astype(str).str.strip() != ""),
    ["apk_name", "version", "apkitself_url", "success"]
].copy()
src.head(), src.shape

(                    apk_name version  \
 0  com.polyverse.bricks.game     122   
 1  com.polyverse.bricks.game     123   
 4       com.punjabimatrimony     335   
 5       com.punjabimatrimony     341   
 6          com.qidafcl.sl054    2207   
 
                                        apkitself_url  success  
 0         https://www.riveroll.top/privacy-policy-2/     True  
 1         https://www.riveroll.top/privacy-policy-2/     True  
 4  https://apps.bharatmatrimony.com/appassuredcon...    False  
 5  https://apps.bharatmatrimony.com/appassuredcon...    False  
 6                https://www.crazylabs.com/apps-pps/     True  ,
 (15, 4))

In [179]:
# 4. 左连接到目标表
merged = left.merge(
    src,
    on=["apk_name", "version"],
    how="left",
    suffixes=("", "_new")
)
merged.head(), merged.shape

(                        apk_name version  \
 0  com.PoxelStudios.CrossyBrakes      27   
 1  com.PoxelStudios.CrossyBrakes      28   
 2   com.QuranReading.quranbangla      25   
 3   com.QuranReading.quranbangla      26   
 4        com.RedLineGames.Game49     114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps/Quran

In [153]:
# mask = merged["success"] == True
# mask.value_counts()

In [180]:
# 6. 更新匹配到的行
mask = merged["success"] == True
merged.loc[mask, "source"] = "apkitself"

In [181]:
mask1 = (merged["success"] != True) &(merged["source"]=="apkitself")
mask1.value_counts()

False    95
True      5
Name: count, dtype: int64

In [182]:
mask1 = (merged["success"] != True) &(merged["source"]=="apkitself")
merged.loc[mask1, "source"] = ""
# merged.loc[mask, "privacy_url"] = merged.loc[mask, "privacy_policy_url"]

merged.head(), merged.shape

(                        apk_name version  \
 0  com.PoxelStudios.CrossyBrakes      27   
 1  com.PoxelStudios.CrossyBrakes      28   
 2   com.QuranReading.quranbangla      25   
 3   com.QuranReading.quranbangla      26   
 4        com.RedLineGames.Game49     114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps/Quran

In [188]:
# 7. 删除临时列
# merged = merged.drop(columns=["privacy_policy_url"])
merged.head(), merged.shape, apk_summary_df_path


(                        apk_name version  \
 0  com.PoxelStudios.CrossyBrakes      27   
 1  com.PoxelStudios.CrossyBrakes      28   
 2   com.QuranReading.quranbangla      25   
 3   com.QuranReading.quranbangla      26   
 4        com.RedLineGames.Game49     114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps/Quran

In [190]:
# 8. 保存回 CSV
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_apkitself_md.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_apkitself_md.csv"
merged.to_csv(apk_summary_df_path, index=False)

不用

In [159]:
# from urllib.parse import urljoin, urlparse
# import re

# def is_google_policy_url(href: str) -> bool:
#     if not href:
#         return True
#     h = href.lower()
#     return (
#         "policies.google.com/privacy" in h
#         or "support.google.com" in h
#         or "myaccount.google.com" in h
#         or "google.com/policies" in h
#     )

# def extract_privacy_policy_url(html: str) -> str | None:
#     try:
#         from bs4 import BeautifulSoup
#         soup = BeautifulSoup(html, "html.parser")

#         # 1) 优先找包含固定提示语的父节点
#         target_text_patterns = [
#             "developer's privacy policy",
#             "developers privacy policy",
#             "for more information about collected and shared data"
#         ]

#         for node in soup.find_all(string=True):
#             txt = " ".join(node.strip().lower().split())
#             if any(p in txt for p in target_text_patterns):
#                 parent = node.parent
#                 if parent:
#                     # 先看当前节点下的链接
#                     for a in parent.find_all("a", href=True):
#                         href = urljoin("https://play.google.com", a["href"].strip())
#                         if not is_google_policy_url(href):
#                             return href

#                     # 再往上找一层
#                     gp = parent.parent
#                     if gp:
#                         for a in gp.find_all("a", href=True):
#                             href = urljoin("https://play.google.com", a["href"].strip())
#                             if not is_google_policy_url(href):
#                                 return href

#         # 2) 回退：找所有文本为 privacy policy 的链接
#         candidates = []
#         for a in soup.find_all("a", href=True):
#             text = " ".join(a.get_text(" ", strip=True).lower().split())
#             href = urljoin("https://play.google.com", a["href"].strip())

#             if "privacy policy" in text:
#                 candidates.append(href)

#         # 3) 优先非 Google policy
#         for href in candidates:
#             if not is_google_policy_url(href):
#                 return href

#         return None

#     except Exception:
#         # fallback regex
#         matches = re.findall(
#             r'href="([^"]+)"[^>]*>\s*privacy\s+policy\s*<',
#             html,
#             flags=re.IGNORECASE
#         )
#         for href in matches:
#             full = urljoin("https://play.google.com", href.strip())
#             if not is_google_policy_url(full):
#                 return full
#         return None

遍历该目录1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json下的所有json文件，文件名前面是apkname，然后是version，得到之后，保存为csv文件

In [160]:
# from pathlib import Path
# import pandas as pd

# SEG_DIR = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json")
# OUT_CSV = SEG_DIR / "apk_versions_summary.csv"

# rows = []
# bad_files = []

# for p in SEG_DIR.glob("*.json"):
#     stem = p.stem  # e.g. com.brave.merge_1513
#     if "_" not in stem:
#         bad_files.append(p.name)
#         continue

#     apk_name, version_str = stem.rsplit("_", 1)

#     rows.append({
#         "apk_name": apk_name,
#         "version": version_str,
#         "json_file": p.name,
#         "source": "from_apk"
#     })

# df = pd.DataFrame(rows)
# df["version_num"] = pd.to_numeric(df["version"], errors="coerce")

# df = df.sort_values(["apk_name", "version_num", "version"], na_position="last").reset_index(drop=True)
# df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

# print("Total json:", len(list(SEG_DIR.glob("*.json"))))
# print("Parsed rows:", len(df))
# print("Bad filenames:", len(bad_files))
# print("Wrote CSV:", OUT_CSV)

加载目录1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json下以versions_summary结尾的csv文件，1
统计目录1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch下以_summary结尾的csv文件，2

对比上面两份文档，找到2中除掉1的内容，保存为apkname_version_google_store_summary.csv文件

In [161]:
# from pathlib import Path
# import pandas as pd

# DIR1 = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json")
# DIR2 = Path(r"1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch")
# OUT = DIR1 / "apkname_version_google_store_summary.csv"

# def load_all_summary_csvs1(folder: Path) -> pd.DataFrame:
#     files = sorted(folder.glob("*versions_summary.csv"))
#     # if not files:
#     #     files = sorted(folder.glob("*summary.csv"))
#     if not files:
#         return pd.DataFrame(columns=["apk_name", "version"])

#     dfs = []
#     for p in files:
#         df = pd.read_csv(p, encoding="utf-8-sig")
#         df["__source_csv__"] = p.name
#         dfs.append(df)

#     out = pd.concat(dfs, ignore_index=True)

#     # 只保留比较需要的列
#     if "apk_name" not in out.columns or "version" not in out.columns:
#         raise ValueError(f"Missing columns in summary csvs under {folder}: need apk_name, version")

#     out["apk_name"] = out["apk_name"].astype(str).str.strip()
#     out["version"] = out["version"].astype(str).str.strip()
#     # out["label"] = "from_google_store"
#     return out

# def load_all_summary_csvs2(folder: Path) -> pd.DataFrame:
#     files = sorted(folder.glob("*_summary.csv"))
#     # if not files:
#     #     files = sorted(folder.glob("*summary.csv"))
#     if not files:
#         return pd.DataFrame(columns=["apk_name", "version"])

#     dfs = []
#     for p in files:
#         df = pd.read_csv(p, encoding="utf-8-sig")
#         df["__source_csv__"] = p.name
#         dfs.append(df)

#     out = pd.concat(dfs, ignore_index=True)

#     # 只保留比较需要的列
#     if "apk_name" not in out.columns or "version" not in out.columns:
#         raise ValueError(f"Missing columns in summary csvs under {folder}: need apk_name, version")

#     out["apk_name"] = out["apk_name"].astype(str).str.strip()
#     out["version"] = out["version"].astype(str).str.strip()
#     # out["label"] = "from_google_store"
#     return out

# df1 = load_all_summary_csvs1(DIR1)
# print(df1.shape)
# df2 = load_all_summary_csvs2(DIR2)
# print(df2.shape)

# set1 = set(zip(df1["apk_name"], df1["version"]))

# mask_in_1 = list(zip(df2["apk_name"], df2["version"]))
# df2_only = df2[~pd.Series(mask_in_1).isin(set1)].copy()

# df2_only = df2_only.sort_values(["apk_name", "version"]).reset_index(drop=True)
# df2_only["label"] = "from_google_store_only"

# # 去掉只出现一次的apkname
# df2_only = df2_only[
#     df2_only.groupby("apk_name")["apk_name"].transform("size") > 1
# ].copy().reset_index(drop=True)

# print(df2_only.shape)


# df2_only.to_csv(OUT, index=False, encoding="utf-8-sig")

# print("DIR1 rows:", len(df1), "unique pairs:", len(set1))
# print("DIR2 rows:", len(df2), "unique pairs:", len(set(zip(df2['apk_name'], df2['version']))))
# print("DIR2 minus DIR1:", len(df2_only))
# print("Wrote:", OUT)